# Notebook Setup

This notebook downloads benchmark datasets, audits categorical features, and creates processed dataset variants.

## Imports

- Load filesystem, numerical, tabular, and progress-bar utilities.
- Load UCI, OpenML, and imbalanced-learn dataset sources.
- Load preprocessing tools used by later cells.

In [9]:
import os
import gc
import sys
import hashlib
import json
from io import BytesIO, StringIO
from urllib.request import urlopen
from zipfile import ZipFile

sys.path.append(os.path.abspath("..")) 

import numpy as np              
import pandas as pd    

from tqdm import tqdm

import re

import openml
import imblearn
from imblearn.datasets import fetch_datasets
from ucimlrepo import fetch_ucirepo


from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder



from utils.load.store import load_datasets_csv, save_dataset_csv
from utils.load.preprocess import standardize_all
from utils.load.create import create_synthetic_overlap

# Dataset Paths and Shared State

Define input and output directories used by all processing variants.

- `RAW_DIR`: downloaded CSV files.
- `PROCESSED_DOWNLOADED_DIR`: standard processed files with categorical encoding.
- `PROCESSED_WITHOUT_C_DOWNLOADED_DIR`: numeric-only files with categorical features removed.
- `RESULTS_METRICS_DIR`: dataset and feature-schema manifests used to audit downstream experiments.

In [10]:
# Define raw-data and processed-output locations shared by preprocessing cells.
RAW_DIR = r"C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\raw"
PROCESSED_DOWNLOADED_DIR = r"C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\preprocessed\downloaded\with_c"
PROCESSED_WITHOUT_C_DOWNLOADED_DIR = r"C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\preprocessed\downloaded\without_c"
RESULTS_METRICS_DIR = r"C:\Users\Maxim\Desktop\re-sc\resc-experiments\results\metrics"
DATASET_MANIFEST_PATH = os.path.join(RESULTS_METRICS_DIR, "dataset_manifest.json")
FEATURE_SCHEMA_PATH = os.path.join(RESULTS_METRICS_DIR, "feature_schema.json")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(RESULTS_METRICS_DIR, exist_ok=True)

# Explicit final benchmark sources make acquisition auditable and repeatable.
IMBLEARN_DATASET_NAMES = [
    "arrhythmia", "car_eval_34", "ecoli", "libras_move", "oil",
    "solar_flare_m0", "spectrometer", "us_crime", "yeast_me2",
]
# Retained OpenML task sources.
OPENML_TASK_IDS = [
    15, 50, 57, 9969,
]
# Re-SC baseline-style binary tasks: rows < 1200 and IR > 1.9.
UCI_BINARY_SPECS = [
    {"dataset_key": "uci_abalone9_18", "source_id": 1, "source_name": "abalone9-18", "minority_labels": ["18"], "majority_labels": ["9"]},
    {"dataset_key": "uci_glass6", "source_id": 42, "source_name": "glass6", "minority_labels": ["7"], "majority_labels": None},
    {"dataset_key": "uci_haberman", "source_id": 43, "source_name": "haberman", "minority_labels": ["2"], "majority_labels": ["1"]},
    {"dataset_key": "uci_page_blocks_13vs4", "source_id": 78, "source_name": "page-blocks-13vs4", "minority_labels": ["3"], "majority_labels": ["2", "5"]},
    {"dataset_key": "uci_spect", "source_id": 95, "source_name": "spect", "minority_labels": ["0"], "majority_labels": ["1"]},
    {"dataset_key": "uci_transfusion", "source_id": 176, "source_name": "transfusion", "minority_labels": ["1"], "majority_labels": ["0"]},
    {"dataset_key": "uci_vertebral", "source_id": 212, "source_name": "vertebral", "minority_labels": ["Normal"], "majority_labels": ["Hernia", "Spondylolisthesis"]},
    {"dataset_key": "uci_wine3", "source_id": 109, "source_name": "wine3", "minority_labels": ["3"], "majority_labels": None},
    {"dataset_key": "uci_wpbc", "source_id": 16, "source_name": "wpbc", "minority_labels": ["R"], "majority_labels": ["N"]},
    {"dataset_key": "uci_yeast_1vs7", "source_id": 110, "source_name": "yeast-1vs7", "minority_labels": ["VAC"], "majority_labels": ["NUC"]},
]
STATIC_UCI_BINARY_SPECS = [
    {"dataset_key": "uci_seeds", "source_id": 236, "source_name": "seeds", "minority_labels": ["1"], "majority_labels": ["2", "3"], "url": "https://archive.ics.uci.edu/static/public/236/seeds.zip", "member": "seeds_dataset.txt", "separator": r"\s+", "n_features": 7},
]
KEEL_BINARY_SPECS = [
    {"dataset_key": "keel_glass0", "source_id": "glass0", "source_name": "glass0", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRlowerThan9/glass0.zip", "member": "glass0.dat", "separator": r"\s*,\s*", "n_features": 9},
    {"dataset_key": "keel_ecoli1", "source_id": "ecoli1", "source_name": "ecoli1", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRlowerThan9/ecoli1.zip", "member": "ecoli1.dat", "separator": r"\s*,\s*", "n_features": 7},
    {"dataset_key": "keel_vowel0", "source_id": "vowel0", "source_name": "vowel0", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p1/vowel0.zip", "member": "vowel0.dat", "separator": r"\s*,\s*", "n_features": 13},
    {"dataset_key": "keel_vehicle0", "source_id": "vehicle0", "source_name": "vehicle0", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRlowerThan9/vehicle0.zip", "member": "vehicle0.dat", "separator": r"\s*,\s*", "n_features": 18},
    {"dataset_key": "keel_cleveland_0vs4", "source_id": "cleveland-0_vs_4", "source_name": "cleveland-0-vs-4", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRhigherThan9p2/cleveland-0_vs_4.zip", "member": "cleveland-0_vs_4.dat", "separator": r"\s*,\s*", "n_features": 13},
    {"dataset_key": "keel_new_thyroid1", "source_id": "new-thyroid1", "source_name": "new-thyroid1", "minority_labels": ["positive"], "majority_labels": ["negative"], "url": "https://sci2s.ugr.es/keel/keel-dataset/datasets/imbalanced/imb_IRlowerThan9/new-thyroid1.zip", "member": "new-thyroid1.dat", "separator": r"\s*,\s*", "n_features": 5},
]
data_list = []

# Dataset Acquisition and Frozen Schema

Fetch explicit imbalanced-learn, UCI, KEEL, and OpenML datasets, build declared binary tasks, save raw CSV files, and persist source metadata.

> The manifest records target mappings, ordered feature types, file hashes, and every acquisition exclusion. Multi-class source labels are collapsed into explicit `minority` and `majority` target values before registration.

In [11]:
# Acquire the frozen candidate corpus and persist its source/schema manifest.
from io import BytesIO, StringIO
from urllib.request import urlopen
from zipfile import ZipFile
data_list = []
dataset_manifest = {"schema_version": 1, "datasets": [], "exclusions": []}
feature_schema = {}

def _python_scalar(value):
    return value.item() if isinstance(value, np.generic) else value

def _sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _record_exclusion(provider, source_id, reason, detail=None):
    exclusion = {"provider": provider, "source_id": str(source_id), "reason": reason}
    if detail:
        exclusion["detail"] = str(detail)
    dataset_manifest["exclusions"].append(exclusion)
    detail_text = f" ({detail})" if detail else ""
    print(f"Excluded {provider} {source_id}: {reason}{detail_text}")

def _register_dataset(dataset_key, provider, source_id, source_name, raw_filename,
                      target_source_name, X, y, features, source_version=None,
                      missing_markers=None, categorical_indicator=None,
                      source_url=None):
    class_counts = pd.Series(y).value_counts(dropna=False)
    # Pandas categoricals may expose zero-count declared levels; ignore them.
    class_counts = class_counts[class_counts > 0]
    if len(class_counts) != 2:
        raise ValueError(f"expected binary target, found {len(class_counts)} classes")

    labels = [_python_scalar(label) for label in class_counts.index.tolist()]
    counts = [int(count) for count in class_counts.tolist()]
    if counts[0] == counts[1]:
        # Balanced ties use a stable label-order convention.
        label_order = sorted(range(2), key=lambda i: str(labels[i]))
        majority_position, minority_position = label_order[0], label_order[1]
    else:
        minority_position = int(np.argmin(counts))
        majority_position = int(np.argmax(counts))
    n_samples, n_features = X.shape
    ir = counts[majority_position] / counts[minority_position]
    n_numeric = sum(feature["semantic_type"] == "numeric" for feature in features)
    if categorical_indicator is None:
        categorical_indicator = [
            feature["semantic_type"] == "categorical" for feature in features
        ]
    feature_schema[dataset_key] = {
        "attribute_names": [feature["name"] for feature in features],
        "categorical_indicator": [bool(value) for value in categorical_indicator],
        "target_name": "target",
        "target_source_name": str(target_source_name),
        "features": features,
    }
    raw_path = os.path.join(RAW_DIR, raw_filename)
    dataset_manifest["datasets"].append({
        "dataset_key": dataset_key,
        "provider": provider,
        "source_id": str(source_id),
        "source_url": source_url,
        "source_version": (
            None if source_version is None else str(source_version)
        ),
        "source_name": str(source_name),
        "raw_filename": raw_filename,
        "sha256": _sha256_file(raw_path),
        "missing_markers": missing_markers or {"pandas_na": True},
        "n_samples": int(n_samples),
        "n_features": int(n_features),
        "feature_schema_ref": dataset_key,
        "categorical_drop_eligible": n_numeric > 0,
        "categorical_drop_exclusion_reason": (
            None if n_numeric > 0 else "no_numeric_predictors"
        ),
        "target": {
            "name": "target",
            "source_name": str(target_source_name),
            "labels": labels,
            "counts": {str(label): count for label, count in zip(labels, counts)},
            "minority_label": labels[minority_position],
            "majority_label": labels[majority_position],
        },
        "eligible": True,
    })
    if n_numeric == 0:
        _record_exclusion(
            provider, source_id, "no_numeric_predictors_for_without_c_track"
        )
    data_list.append({
        "Dataset": str(source_name).replace("_", "\\_"),
        "Samples": int(n_samples),
        "Features": int(n_features),
        "Minority": counts[minority_position],
        "Majority": counts[majority_position],
        "IR": round(ir, 2),
    })

def _label_text(value):
    if pd.isna(value):
        return None
    if isinstance(value, (float, np.floating)) and float(value).is_integer():
        return str(int(value))
    return str(value).strip()

def _restrict_binary_task(X, y, spec):
    X = pd.DataFrame(X).copy().reset_index(drop=True)
    X.columns = [str(column) for column in X.columns]
    y = pd.Series(y).reset_index(drop=True)
    labels = y.map(_label_text)
    minority_mask = labels.isin(spec["minority_labels"])
    declared_majority = spec["majority_labels"]
    if declared_majority is None:
        majority_mask = labels.notna() & ~minority_mask
    else:
        majority_mask = labels.isin(declared_majority)
    if not minority_mask.any() or not majority_mask.any():
        raise ValueError("declared binary labels were not found")
    mask = minority_mask | majority_mask
    binary_target = pd.Series(
        np.where(minority_mask.loc[mask], "minority", "majority"),
        name=y.name or "target",
    )
    return X.loc[mask].reset_index(drop=True), binary_target.reset_index(drop=True)

def _infer_features(X, provenance):
    return [
        {
            "name": column,
            "position": index,
            "semantic_type": (
                "numeric" if pd.api.types.is_numeric_dtype(X[column]) else "categorical"
            ),
            "provenance": provenance,
        }
        for index, column in enumerate(X.columns)
    ]

def _save_and_register_binary_task(spec, provider, source_id, source_name, target_name, X, y,
                                   features, source_version, source_url, categorical_indicator=None):
    raw_filename = f"{spec['dataset_key']}.csv"
    raw = X.copy()
    raw["target"] = y
    raw.to_csv(os.path.join(RAW_DIR, raw_filename), index=False)
    _register_dataset(
        spec["dataset_key"], provider, source_id, source_name, raw_filename,
        target_name, X, y, features, source_version=source_version,
        missing_markers={"pandas_na": True},
        categorical_indicator=categorical_indicator, source_url=source_url,
    )

def _read_zipped_binary_source(spec):
    with urlopen(spec["url"], timeout=60) as response:
        payload = response.read()
    with ZipFile(BytesIO(payload)) as archive:
        text = archive.read(spec["member"]).decode("utf-8")
    data_lines = [
        line for line in text.splitlines()
        if line.strip() and not line.lstrip().startswith("@")
    ]
    table = pd.read_csv(
        StringIO("\n".join(data_lines)), header=None, sep=spec["separator"],
        engine="python",
    )
    expected_columns = int(spec["n_features"]) + 1
    if table.shape[1] != expected_columns:
        raise ValueError(
            f"expected {expected_columns} columns, found {table.shape[1]}"
        )
    X = table.iloc[:, :-1].copy()
    X.columns = [f"feature_{index}" for index in range(X.shape[1])]
    y = table.iloc[:, -1].copy()
    return X, y

for spec in tqdm(UCI_BINARY_SPECS, desc="Fetching UCI binary tasks"):
    try:
        dataset = fetch_ucirepo(id=int(spec["source_id"]))
        X = dataset.data.features
        y = dataset.data.targets
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]
        X, y = _restrict_binary_task(X, y, spec)
        _save_and_register_binary_task(
            spec, "UCI", spec["source_id"], spec["source_name"],
            y.name or "target", X, y, _infer_features(X, "uci_dtype"),
            None, f"https://archive.ics.uci.edu/dataset/{spec['source_id']}",
        )
    except Exception as exc:
        _record_exclusion("UCI", spec["source_id"], "acquisition_error", exc)

for spec in tqdm(STATIC_UCI_BINARY_SPECS, desc="Fetching static UCI binary tasks"):
    try:
        X, y = _read_zipped_binary_source(spec)
        X, y = _restrict_binary_task(X, y, spec)
        _save_and_register_binary_task(
            spec, "UCI", spec["source_id"], spec["source_name"], "class",
            X, y, _infer_features(X, "uci_static_zip"), None, spec["url"],
        )
    except Exception as exc:
        _record_exclusion("UCI", spec["source_id"], "acquisition_error", exc)

for spec in tqdm(KEEL_BINARY_SPECS, desc="Fetching KEEL binary tasks"):
    try:
        X, y = _read_zipped_binary_source(spec)
        X, y = _restrict_binary_task(X, y, spec)
        _save_and_register_binary_task(
            spec, "KEEL", spec["source_id"], spec["source_name"], "Class",
            X, y, _infer_features(X, "keel_dat"), None, spec["url"],
        )
    except Exception as exc:
        _record_exclusion("KEEL", spec["source_id"], "acquisition_error", exc)

imb_datasets = fetch_datasets()
for name in tqdm(IMBLEARN_DATASET_NAMES, desc="Fetching imbalanced-learn datasets"):
    if name not in imb_datasets:
        _record_exclusion("imbalanced-learn", name, "candidate_not_available")
        continue
    try:
        dataset = imb_datasets[name]
        X, y = dataset.data, dataset.target
        n_features = X.shape[1]
        feature_names = [f"feature_{i}" for i in range(n_features)]
        features = [
            {"name": feature_name, "position": i, "semantic_type": "numeric",
             "provenance": "imblearn_numeric_contract"}
            for i, feature_name in enumerate(feature_names)
        ]
        df = pd.DataFrame(X, columns=feature_names)
        df["target"] = y
        raw_filename = f"imblearn_{name}.csv"
        df.to_csv(os.path.join(RAW_DIR, raw_filename), index=False)
        _register_dataset(
            f"imblearn_{name}", "imbalanced-learn", name, name, raw_filename,
            "target", df[feature_names], y, features,
            source_version=imblearn.__version__,
            missing_markers={"pandas_na": True},
            source_url="imbalanced-learn.fetch_datasets",
        )
    except Exception as exc:
        _record_exclusion("imbalanced-learn", name, "acquisition_error", exc)

for task_id in tqdm(OPENML_TASK_IDS, desc="Fetching OpenML tasks"):
    try:
        task = openml.tasks.get_task(int(task_id))
        target_name = task.target_name
        if not target_name:
            raise ValueError("task has no declared target_name")
        dataset_id = int(task.dataset_id)
        dataset = openml.datasets.get_dataset(dataset_id)
        X, y, categorical_indicator, attribute_names = dataset.get_data(
            dataset_format="dataframe", target=target_name
        )
        n_features = X.shape[1]
        if len(attribute_names) != n_features or len(categorical_indicator) != n_features:
            raise ValueError("OpenML feature names/types do not match X columns")
        ordered_names = [str(name) for name in attribute_names]
        X = X.copy()
        X.columns = ordered_names
        features = [
            {"name": feature_name, "position": i,
             "semantic_type": "categorical" if bool(is_cat) else "numeric",
             "provenance": "openml_categorical_indicator"}
            for i, (feature_name, is_cat) in enumerate(zip(ordered_names, categorical_indicator))
        ]
        dataset_slug = re.sub(r"[^A-Za-z0-9.-]+", "_", str(dataset.name)).strip("_")
        raw_filename = f"openml_task_{int(task_id)}_{dataset_slug}.csv"
        df_openml = X.copy()
        df_openml["target"] = y
        df_openml.to_csv(os.path.join(RAW_DIR, raw_filename), index=False)
        _register_dataset(
            f"openml_task_{int(task_id)}_{dataset_slug}", "OpenML",
            f"task:{int(task_id)};dataset:{dataset_id}", str(dataset.name), raw_filename,
            target_name, X, y, features,
            source_version=getattr(dataset, "version", None),
            missing_markers={
                "pandas_na": True,
                "openml_default_missing_value": getattr(
                    dataset, "default_missing_value", None
                ),
            },
            categorical_indicator=categorical_indicator,
            source_url=f"https://www.openml.org/t/{int(task_id)}",
        )
    except Exception as exc:
        _record_exclusion("OpenML", task_id, "acquisition_error", exc)

with open(DATASET_MANIFEST_PATH, "w", encoding="utf-8") as handle:
    json.dump(dataset_manifest, handle, indent=2, ensure_ascii=False)
with open(FEATURE_SCHEMA_PATH, "w", encoding="utf-8") as handle:
    json.dump(feature_schema, handle, indent=2, ensure_ascii=False)

_manifest_by_filename = {
    entry["raw_filename"]: entry for entry in dataset_manifest["datasets"]
}

def load_raw_dataset(filename):
    entry = _manifest_by_filename[filename]
    schema = feature_schema[entry["feature_schema_ref"]]
    categorical_columns = [
        feature["name"] for feature in schema["features"]
        if feature["semantic_type"] == "categorical"
    ]
    dtype = {column: "string" for column in categorical_columns}
    return pd.read_csv(os.path.join(RAW_DIR, filename), dtype=dtype)

def encode_minority_target(y, entry):
    """Encode the manifest majority as 0 and minority as 1."""
    majority = _label_text(entry["target"]["majority_label"])
    minority = _label_text(entry["target"]["minority_label"])
    labels = pd.Series(y).map(_label_text)
    recognized = labels.eq(majority) | labels.eq(minority)
    if labels.isna().any() or not recognized.all():
        observed = sorted(labels.dropna().unique().tolist())
        raise ValueError(
            f"Observed target labels do not match manifest: {observed}"
        )
    return labels.eq(minority).to_numpy(dtype=int)

print(f"Acquired {len(dataset_manifest['datasets'])} datasets; "
      f"recorded {len(dataset_manifest['exclusions'])} exclusions.")


Fetching OpenML tasks: 100%|██████████| 4/4 [00:00<00:00, 53.69it/s]

Acquired 30 datasets; recorded 0 exclusions.


# Categorical Feature Audit

Inspect each downloaded CSV before preprocessing. The cell reports total feature count, schema-declared categorical-feature count, and unique-value count per categorical column.

- Target column excluded from feature audit.
- Missing values excluded from cardinality counts.
- Provider metadata is used because CSV dtypes cannot identify numeric-coded categorical features.

In [12]:
# Inspect schema-declared categorical columns and their cardinalities.
print(f"Inspecting categorical features in downloaded datasets: {RAW_DIR}\n")

for filename in sorted(os.listdir(RAW_DIR)):
    if not filename.lower().endswith(".csv"):
        continue

    if filename not in _manifest_by_filename:
        print(f"Skipping unmanifested raw file: {filename}")
        continue
    df = load_raw_dataset(filename)
    entry = _manifest_by_filename[filename]
    schema = feature_schema[entry["feature_schema_ref"]]
    total_features = len(schema["features"])
    categorical_columns = [
        feature["name"] for feature in schema["features"]
        if feature["semantic_type"] == "categorical"
    ]
    values_per_feature = df[categorical_columns].nunique(dropna=True).to_numpy(dtype=int)

    print(
        f"{filename}: {total_features} total features; "
        f"{len(categorical_columns)} categorical features; "
        f"values per categorical feature = {values_per_feature}"
    )


Inspecting categorical features in downloaded datasets: C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\raw

imblearn_arrhythmia.csv: 278 total features; 0 categorical features; values per categorical feature = []
imblearn_car_eval_34.csv: 21 total features; 0 categorical features; values per categorical feature = []
imblearn_ecoli.csv: 7 total features; 0 categorical features; values per categorical feature = []
imblearn_libras_move.csv: 90 total features; 0 categorical features; values per categorical feature = []
imblearn_oil.csv: 49 total features; 0 categorical features; values per categorical feature = []
imblearn_solar_flare_m0.csv: 32 total features; 0 categorical features; values per categorical feature = []
imblearn_spectrometer.csv: 93 total features; 0 categorical features; values per categorical feature = []
imblearn_us_crime.csv: 100 total features; 0 categorical features; values per categorical feature = []
imblearn_yeast_me2.csv: 8 total features; 0 categorical featu

# Numeric-Only Preprocessing

This workflow creates a second processed-dataset variant for experiments that exclude categorical features.

## Workflow

1. Read each manifest-backed CSV file from `RAW_DIR`.
2. Separate feature columns from the target column.
3. Select schema-declared numeric features and drop categorical features.
4. Impute missing numeric values with the median.
5. Standardize retained numeric features.
6. Encode the manifest majority as `0` and minority as `1`, then save the result.

## Output

Files are written to `PROCESSED_WITHOUT_C_DOWNLOADED_DIR`, preserving original filenames.
Datasets with no numeric predictors are skipped for this categorical-drop track and remain eligible for the full OHE track.

In [13]:
# Build numeric-only preprocessing outputs using the frozen feature schema.
os.makedirs(PROCESSED_WITHOUT_C_DOWNLOADED_DIR, exist_ok=True)
print(f"Reading raw datasets from: {RAW_DIR}")
print(f"Saving numeric-only datasets to: {PROCESSED_WITHOUT_C_DOWNLOADED_DIR}\n")

for filename in sorted(os.listdir(RAW_DIR)):
    if not filename.lower().endswith(".csv"):
        continue

    # Load one manifest-backed dataset and separate features from target.
    if filename not in _manifest_by_filename:
        print(f"Skipping unmanifested raw file: {filename}")
        continue
    entry = _manifest_by_filename[filename]
    schema = feature_schema[entry["feature_schema_ref"]]
    file_path = os.path.join(RAW_DIR, filename)
    save_path = os.path.join(PROCESSED_WITHOUT_C_DOWNLOADED_DIR, filename)

    try:
        df = load_raw_dataset(filename)
        target_col = schema["target_name"]
        X = df.drop(columns=[target_col])
        y = df[target_col]

        # Keep only schema-declared numeric features; categorical columns are discarded.
        numeric_columns = [
            feature["name"] for feature in schema["features"]
            if feature["semantic_type"] == "numeric"
        ]
        X_numeric = X[numeric_columns]

        # Impute missing numeric values and standardize retained features.
        if X_numeric.shape[1] > 0:
            numeric_pipeline = Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler())
            ])
            X_processed = numeric_pipeline.fit_transform(X_numeric)
            df_processed = pd.DataFrame(
                X_processed, columns=X_numeric.columns, index=df.index
            )
        else:
            # The categorical-drop track excludes datasets with no numeric predictors.
            print(f"Skipping categorical-drop track (no numeric predictors): {filename}")
            continue

        # Encode majority as 0 and minority as 1 using the manifest mapping.
        df_processed["target"] = encode_minority_target(y, entry)
        df_processed.to_csv(save_path, index=False)
        print(f"Successfully preprocessed without categorical features: {filename}")

    except Exception as e:
        print(f"Error processing {filename} without categorical features: {e}")


Reading raw datasets from: C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\raw
Saving numeric-only datasets to: C:\Users\Maxim\Desktop\re-sc\resc-experiments\data\preprocessed\downloaded\without_c

Successfully preprocessed without categorical features: imblearn_arrhythmia.csv
Successfully preprocessed without categorical features: imblearn_car_eval_34.csv
Successfully preprocessed without categorical features: imblearn_ecoli.csv
Successfully preprocessed without categorical features: imblearn_libras_move.csv
Successfully preprocessed without categorical features: imblearn_oil.csv
Successfully preprocessed without categorical features: imblearn_solar_flare_m0.csv
Successfully preprocessed without categorical features: imblearn_spectrometer.csv
Successfully preprocessed without categorical features: imblearn_us_crime.csv
Successfully preprocessed without categorical features: imblearn_yeast_me2.csv
Successfully preprocessed without categorical features: keel_cleveland_0vs4.csv
Succes

# Full Feature Preprocessing

Create the main processed dataset variant. Manifest-declared numeric features are imputed and standardized; manifest-declared categorical features are imputed and one-hot encoded.

Output is written to `PROCESSED_DOWNLOADED_DIR` with majority targets encoded as `0` and minority targets as `1`.

In [14]:
# Preprocess raw datasets with schema-driven numeric and categorical branches.
os.makedirs(PROCESSED_DOWNLOADED_DIR, exist_ok=True)

for filename in sorted(os.listdir(RAW_DIR)):
    if not filename.endswith(".csv"):
        continue
    if filename not in _manifest_by_filename:
        print(f"Skipping unmanifested raw file: {filename}")
        continue
    entry = _manifest_by_filename[filename]
    schema = feature_schema[entry["feature_schema_ref"]]
        
    file_path = os.path.join(RAW_DIR, filename)
    save_path = os.path.join(PROCESSED_DOWNLOADED_DIR, filename)
    
    try:
        df = load_raw_dataset(filename)
        target_col = schema["target_name"]
        
        X = df.drop(columns=[target_col])
        y = df[target_col]
        
        num_cols = [
            feature["name"] for feature in schema["features"]
            if feature["semantic_type"] == "numeric"
        ]
        cat_cols = [
            feature["name"] for feature in schema["features"]
            if feature["semantic_type"] == "categorical"
        ]
        # Convert pandas StringDtype missing values to np.nan for sklearn imputers.
        for column in cat_cols:
            X[column] = X[column].map(
                lambda value: np.nan if pd.isna(value) else str(value)
            ).astype(object)
        
        num_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ])
        
        cat_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ])
        
        preprocessor = ColumnTransformer(
            transformers=[
                ('num', num_pipeline, num_cols),
                ('cat', cat_pipeline, cat_cols)
            ],
            remainder='drop'
        )
        
        X_processed = preprocessor.fit_transform(X)
        
        feature_names = []
        if num_cols:
            feature_names.extend(num_cols)
        if cat_cols:
            cat_encoder = preprocessor.named_transformers_['cat'].named_steps['ohe']
            cat_features = cat_encoder.get_feature_names_out(cat_cols)
            feature_names.extend(cat_features)
            
        df_processed = pd.DataFrame(X_processed, columns=feature_names)
        
        # Encode majority as 0 and minority as 1 using the manifest mapping.
        df_processed['target'] = encode_minority_target(y, entry)
        
        df_processed.to_csv(save_path, index=False)
        print(f"Successfully preprocessed: {filename}")
        
        del df
        del X
        del y
        del X_processed
        del df_processed
        gc.collect()
        
    except Exception as e:
        print(f"Error processing {filename}: {e}")

Successfully preprocessed: imblearn_arrhythmia.csv
Successfully preprocessed: imblearn_car_eval_34.csv
Successfully preprocessed: imblearn_ecoli.csv
Successfully preprocessed: imblearn_libras_move.csv
Successfully preprocessed: imblearn_oil.csv
Successfully preprocessed: imblearn_solar_flare_m0.csv
Successfully preprocessed: imblearn_spectrometer.csv
Successfully preprocessed: imblearn_us_crime.csv
Successfully preprocessed: imblearn_yeast_me2.csv
Successfully preprocessed: keel_cleveland_0vs4.csv
Successfully preprocessed: keel_ecoli1.csv
Successfully preprocessed: keel_glass0.csv
Successfully preprocessed: keel_new_thyroid1.csv
Successfully preprocessed: keel_vehicle0.csv
Successfully preprocessed: keel_vowel0.csv
Successfully preprocessed: openml_task_15_breast-w.csv
Successfully preprocessed: openml_task_50_heart-h.csv
Successfully preprocessed: openml_task_57_ionosphere.csv
Successfully preprocessed: openml_task_9969_thoracic-surgery.csv
Successfully preprocessed: uci_abalone9_18.

# One-Hot-Encoded Dataset Summary

Build the paper table directly from the current raw files and one-hot-encoded preprocessing outputs. No feature or class counts are hard-coded.

- `Features before preprocessing` counts predictors in each raw CSV.
- `Features after preprocessing` counts predictors in the corresponding file under `PROCESSED_DOWNLOADED_DIR`.
- Sample and class counts are calculated from the raw file using the manifest's minority-label definition.

In [15]:
# Assemble one file-derived row per manifest dataset.
summary_rows = []

for entry in dataset_manifest["datasets"]:
    filename = entry["raw_filename"]
    raw_path = os.path.join(RAW_DIR, filename)
    processed_path = os.path.join(PROCESSED_DOWNLOADED_DIR, filename)

    missing_files = [
        path for path in (raw_path, processed_path) if not os.path.exists(path)
    ]
    if missing_files:
        print(
            f"Skipping {entry['dataset_key']}: missing required file(s) "
            f"{missing_files}"
        )
        continue

    schema = feature_schema[entry["feature_schema_ref"]]
    target_column = schema["target_name"]
    raw_frame = load_raw_dataset(filename)
    processed_columns = pd.read_csv(processed_path, nrows=0).columns

    features_before = sum(column != target_column for column in raw_frame.columns)
    features_after = sum(column != "target" for column in processed_columns)
    encoded_target = encode_minority_target(raw_frame[target_column], entry)
    minority_count = int((encoded_target == 1).sum())
    majority_count = int((encoded_target == 0).sum())

    if features_before != int(entry["n_features"]):
        raise ValueError(f"Raw feature-count mismatch for {entry['dataset_key']}")
    if len(raw_frame) != minority_count + majority_count:
        raise ValueError(f"Class-count mismatch for {entry['dataset_key']}")

    dataset_name = str(entry["source_name"]).replace("_", "\\_")

    summary_rows.append({
        "Dataset": dataset_name,
        "Samples": len(raw_frame),
        "Features before preprocessing": features_before,
        "Features after preprocessing": features_after,
        "Minority": minority_count,
        "Majority": majority_count,
        "IR": round(majority_count / minority_count, 2),
    })

df_datasets = pd.DataFrame(summary_rows)
df_datasets = df_datasets.sort_values(
    by=["IR", "Dataset"]
).reset_index(drop=True)
df_datasets

,Dataset,Samples,Features before preprocessing,Features after preprocessing,Minority,Majority,IR
0,heart-h,294,13,25,106,188,1.77
1,ionosphere,351,34,34,126,225,1.79
2,breast-w,699,9,9,241,458,1.90
3,seeds,210,7,7,70,140,2.00
4,glass0,214,9,9,70,144,2.06
5,vertebral,310,6,6,100,210,2.10
6,wine3,178,13,13,48,130,2.71
7,haberman,306,3,3,81,225,2.78
8,transfusion,748,4,4,178,570,3.20
9,wpbc,198,33,33,47,151,3.21


# LaTeX Table Output

Generate a complete one-column LaTeX float. It can replace Table~`tab:benchmark_datasets` directly in `main.tex`.

In [16]:
latex_frame = df_datasets.rename(columns={
    "Features before preprocessing": r"$d_{\mathrm{before}}$",
    "Features after preprocessing": r"$d_{\mathrm{after}}$",
})
tabular_latex = latex_frame.to_latex(
    index=False,
    escape=False,
    float_format=lambda value: f"{value:.2f}",
    column_format=r"@{}lrrrrrr@{}",
)
dataset_table_latex = "\n".join([
    r"\begin{table}[!t]",
    r"\centering",
    (r"\caption{Characteristics of the benchmark imbalanced datasets. "
     r"$d_{\mathrm{before}}$ and $d_{\mathrm{after}}$ denote feature "
     r"counts before and after categorical predictors are one-hot encoded.}"),
    r"\label{tab:benchmark_datasets}",
    r"\renewcommand{\arraystretch}{0.95}",
    r"\setlength{\tabcolsep}{1.5pt}",
    r"\scriptsize",
    r"\resizebox{\columnwidth}{!}{%",
    tabular_latex.rstrip(),
    r"}",
    r"\end{table}",
])
print(dataset_table_latex)

\begin{table}[!t]
\centering
\caption{Characteristics of the benchmark imbalanced datasets. $d_{\mathrm{before}}$ and $d_{\mathrm{after}}$ denote feature counts before and after categorical predictors are one-hot encoded.}
\label{tab:benchmark_datasets}
\renewcommand{\arraystretch}{0.95}
\setlength{\tabcolsep}{1.5pt}
\scriptsize
\resizebox{\columnwidth}{!}{%
\begin{tabular}{@{}lrrrrrr@{}}
\toprule
Dataset & Samples & $d_{\mathrm{before}}$ & $d_{\mathrm{after}}$ & Minority & Majority & IR \\
\midrule
heart-h & 294 & 13 & 25 & 106 & 188 & 1.77 \\
ionosphere & 351 & 34 & 34 & 126 & 225 & 1.79 \\
breast-w & 699 & 9 & 9 & 241 & 458 & 1.90 \\
seeds & 210 & 7 & 7 & 70 & 140 & 2.00 \\
glass0 & 214 & 9 & 9 & 70 & 144 & 2.06 \\
vertebral & 310 & 6 & 6 & 100 & 210 & 2.10 \\
wine3 & 178 & 13 & 13 & 48 & 130 & 2.71 \\
haberman & 306 & 3 & 3 & 81 & 225 & 2.78 \\
transfusion & 748 & 4 & 4 & 178 & 570 & 3.20 \\
wpbc & 198 & 33 & 33 & 47 & 151 & 3.21 \\
vehicle0 & 846 & 18 & 18 & 199 & 647 & 3.25 \\
ec